***IMPORTS , importing used libraries***

In [1]:
!pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-core faiss-cpu

In [40]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
import re

In [2]:
#helper functions

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"


def add_line_numbers(code: str) -> str:
    return "\n".join(
        f"{i+1:4} | {line}"
        for i, line in enumerate(code.splitlines())
    )

def review_python_file(file_path: str):
    # Read the file
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    # Add line numbers
    numbered_code = add_line_numbers(code)

    # Run the AI review
    response = chain.invoke(numbered_code)

    return response

***loading files from cheatsheets directory***

In [3]:
cheatsheets_path = "/kaggle/input/datasets/yassinharraz/cheatsheet"
loader = DirectoryLoader(
    cheatsheets_path,
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

In [4]:
len(documents)

121

In [5]:
#documents[77]

***now we will choose and implement chunking size and method***

In [6]:
#creating tet splitter to split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [7]:
len(chunks)

6125

In [8]:
print(chunks[0].page_content)

# Error Handling Cheat Sheet

## Introduction

Error handling is a part of the overall security of an application. Except in movies, an attack always begins with a **Reconnaissance** phase in which the attacker will try to gather as much technical information (often *name* and *version* properties) as possible about the target, such as the application server, frameworks, libraries, etc.

Unhandled errors can assist an attacker in this initial phase, which is very important for the rest of the attack.


***now we will embedd chunks and store in   FAISS***

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
## storing embedding in faais vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
type(vector_store)

langchain_community.vectorstores.faiss.FAISS

In [11]:
vector_store.index.ntotal

6125

**implementing retrieval part**

In [12]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
#verfiying it works
query = "How should Python variables be named?"
results = retriever.invoke(query)
len(results)

3

In [13]:
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("=" * 60)
    print(doc.metadata["source"])
    print()
    print(doc.page_content)
    print("\n")

Result 1
/kaggle/input/datasets/yassinharraz/cheatsheet/cheatsheets/pep08.md

Function and Variable Names
Function names should be lowercase, with words separated by underscores as necessary to improve readability.

Variable names follow the same convention as function names.

mixedCase is allowed only in contexts where that’s already the prevailing style (e.g. threading.py), to retain backwards compatibility.

Function and Method Arguments
Always use self for the first argument to instance methods.

Always use cls for the first argument to class methods.


Result 2
/kaggle/input/datasets/yassinharraz/cheatsheet/cheatsheets/pep08.md

Global Variable Names
(Let’s hope that these variables are meant for use inside one module only.) The conventions are about the same as those for functions.

Modules that are designed for use via from M import * should use the __all__ mechanism to prevent exporting globals, or use the older convention of prefixing such globals with an underscore (which you

In [14]:
#Qwen/Qwen2.5-7B-Instruct
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [15]:
response = llm.invoke(
    "Explain what SQL Injection is in two sentences."
)

print(response)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 SQL Injection is a code injection technique that attackers use to exploit vulnerabilities in database management systems by inserting malicious SQL statements through application inputs. This can allow them to manipulate queries, access, modify, or delete data from the database.


In [37]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert Python code reviewer.

Your job is to review Python code using ONLY the provided documentation.

Review the code for:

- PEP8 violations
- Clean Code issues
- Security vulnerabilities
- Bugs
- Performance problems

For every issue you find:

- Explain why it is a problem.
- Suggest a fix.

If there are no issues, clearly state that no issues were found.

Return your response in a structured format.
"""
        ),
        (
            "human",
            """
Documentation:

{context}

----------------------------------------

Python Code:

{code}

----------------------------------------

Review this code.

Return your response using the following format:

{format_instructions}
"""
        ),
    ]
)

In [23]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [34]:
#output schema
from pydantic import BaseModel, Field

class Issue(BaseModel):
    severity: str = Field(
        description="Severity level: Low, Medium, or High"
    )

    line: int = Field(
        description="Line number where the issue occurs"
    )

    issue: str = Field(
        description="Short title of the detected issue"
    )

    explanation: str = Field(
        description="Detailed explanation of why this is a problem"
    )

    suggested_fix: str = Field(
        description="Recommended way to fix the issue"
    )


from typing import List

class CodeReviewReport(BaseModel):
    file: str = Field(
        description="Name of the analyzed Python file"
    )

    issues: List[Issue] = Field(
        description="List of detected issues"
    )

from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(
    pydantic_object=CodeReviewReport
) ##parser that  convert the LLM's output into a CodeReviewReport

In [58]:
chain = (
    {
        "context": retriever | format_docs,
        "code": RunnablePassthrough(),
        "format_instructions": RunnableLambda(
            lambda _: parser.get_format_instructions()
        ),
    }
    | prompt
    | llm
    | parser
    #| RunnableLambda(lambda x: x.model_dump_json(indent=2))    
)

In [59]:
sample_code = """
def GetUser(ID):
    query = "SELECT * FROM users WHERE id=" + ID
    return query
"""

response = chain.invoke(sample_code)

print(response)
#print(response.model_dump_json(indent=2))


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


file='unknown.py' issues=[Issue(severity='High', line=1, issue='SQL Injection Vulnerability', explanation='The query string is constructed by concatenating user input directly into the SQL statement, which makes the application vulnerable to SQL injection attacks.', suggested_fix='Use parameterized queries to safely include user input in the SQL statement.'), Issue(severity='Medium', line=1, issue='Potential Bug', explanation='The function does not validate or sanitize the input before using it in the SQL query, which could lead to unexpected behavior or errors if the input is not a valid integer.', suggested_fix='Add input validation to ensure that the input is a valid integer before constructing the query.')]
